In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from feature_eng import preprocess
df = pd.read_csv("~/ML/Data/house_prices/train.csv")

x_train,x_test = train_test_split(df,test_size=0.2,random_state=42)

In [5]:
x_train,x_test,y_train,y_test = preprocess(x_train,x_test)

In [6]:
x_train.isna().mean()
x_test.isna().mean()

LotFrontage              0.0
LotArea                  0.0
Street                   0.0
LotShape                 0.0
Utilities                0.0
                        ... 
SaleCondition_AdjLand    0.0
SaleCondition_Alloca     0.0
SaleCondition_Family     0.0
SaleCondition_Normal     0.0
SaleCondition_Partial    0.0
Length: 188, dtype: float64

In [24]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV,KFold,cross_val_score
from sklearn.pipeline import Pipeline

kf = KFold(n_splits=5,random_state=42,shuffle=True)

param_grid = {
    'n_estimators': range(100,500,100),
    'max_depth': range(1,10,1),
    'min_samples_split': range(2,20,2),
    'min_samples_leaf': range(1,10,1),
    'criterion': ['squared_error'],
}

grid_search = RandomizedSearchCV(
    RandomForestRegressor(),
    param_distributions=param_grid,   # note: param_distributions, not param_grid
    n_iter=10,                        # goes here
    cv=kf,
    scoring='neg_root_mean_squared_error',
    verbose=2,
    return_train_score=True,
    random_state=42
)


In [25]:
result = grid_search.fit(x_train,y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END criterion=squared_error, max_depth=3, min_samples_leaf=6, min_samples_split=18, n_estimators=100; total time=   0.4s
[CV] END criterion=squared_error, max_depth=3, min_samples_leaf=6, min_samples_split=18, n_estimators=100; total time=   0.4s
[CV] END criterion=squared_error, max_depth=3, min_samples_leaf=6, min_samples_split=18, n_estimators=100; total time=   0.4s
[CV] END criterion=squared_error, max_depth=3, min_samples_leaf=6, min_samples_split=18, n_estimators=100; total time=   0.4s
[CV] END criterion=squared_error, max_depth=3, min_samples_leaf=6, min_samples_split=18, n_estimators=100; total time=   0.4s
[CV] END criterion=squared_error, max_depth=4, min_samples_leaf=9, min_samples_split=18, n_estimators=300; total time=   1.4s
[CV] END criterion=squared_error, max_depth=4, min_samples_leaf=9, min_samples_split=18, n_estimators=300; total time=   1.3s
[CV] END criterion=squared_error, max_depth=4, min_sample

In [1]:
best_idx = result.best_index_
train_rmse = -result.cv_results_['mean_train_score'][best_idx]
val_rmse = -result.best_score_
val_std = result.cv_results_['std_test_score'][best_idx]

print(train_rmse, val_rmse, val_std)

NameError: name 'result' is not defined

In [32]:
import dagshub
import mlflow
dagshub.init(repo_owner='ldavi22', repo_name='cs229-231', mlflow=True)

mlflow.set_tracking_uri("https://dagshub.comp/ldavi22/cs229-231.mlflow")
mlflow.set_experiment('house-prices-linear-regression_')

with mlflow.start_run(run_name="Random Forrest Regressor"):

    best_idx = result.best_index_
    train_rmse = -result.cv_results_['mean_train_score'][best_idx]
    val_rmse = -result.best_score_
    val_std = result.cv_results_['std_test_score'][best_idx]

    mlflow.log_param("model", "RandomForrestRegressor")
    mlflow.log_param("n_features", x_train.shape[1])
    mlflow.log_param("n_samples", x_train.shape[0])
    mlflow.log_param("feature_selection", "RandomForestRegressor")

    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("val_rmse", val_rmse)
    mlflow.log_metric("val_std", val_std)
    mlflow.log_metric("overfit_gap", val_rmse - train_rmse)

    for key,value in result.best_params_.items():
        mlflow.log_param(key, value)


Initialized MLflow to track repo "ldavi22/cs229-231"

Repository ldavi22/cs229-231 initialized!

🏃 View run Random Forrest Regressor at: https://dagshub.com/ldavi22/cs229-231.mlflow/#/experiments/1/runs/f0574a0d3e07496aac48b0928f605cf1
🧪 View experiment at: https://dagshub.com/ldavi22/cs229-231.mlflow/#/experiments/1
